In [7]:
"""
================================================================================
t-SNE降维 + 主动学习迭代演化可视化
================================================================================
输出图片：
    tSNE_00_perplexity_comparison.png  ← 困惑度对比图（选参依据）
    tSNE_01_main_scatter.png           ← 主散点图
    tSNE_02_density_evolution.png      ← 密度演化图
    tSNE_03_FINAL.png                  ← 合并版（论文用）
    tSNE_04_KQ_violin.png              ← KQ小提琴图（新增）
    tSNE_05_feature_heatmap.png        ← 特征分布热力图（新增）

输出Excel：
    data_01_tSNE_coordinates.xlsx          ← 坐标+KQ（对应图01/02/03）
    data_02_centroids.xlsx                 ← 各轮重心（对应图01/03）
    data_03_KDE_grid_origin.xlsx           ← 原始KDE网格（对应图01/03背景）
    data_04_KDE_grid_round0to5.xlsx        ← 各轮KDE网格（对应图02/03下半）
    data_05_perplexity_comparison.xlsx     ← 困惑度对比数据（对应图00）
    data_06_KQ_violin.xlsx                 ← KQ分布数据（对应图04）新增
    data_07_feature_heatmap.xlsx           ← 特征分布数据（对应图05）新增
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.stats import gaussian_kde
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from matplotlib.lines import Line2D
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['mathtext.fontset'] = 'dejavusans'

# ============================================================
# 配置参数（完全不变）
# ============================================================
INPUT_FILE = (
    r"D:\博一下\第一章主动学习\主动学习结果_v7.6_方案D_LogEI_SHAP全程引导版"
    r"\用于UMAP降维+密度演化图-4.8.xlsx"
)
OUTPUT_DIR = (
    r"D:\博一下\第一章主动学习\主动学习结果_v7.6_方案D_LogEI_SHAP全程引导版"
    r"\迭代可视化_tSNE2"
)

FEATURE_COLS = [
    'x',
    'ΔSmix',
    'Ω',
    'Ec',
    'deltaP_E13 Electron affinity',
    'deltaP_热导率W/(mk)',
]

FEATURE_SHORT = {
    'x':                            'χ',
    'ΔSmix':                        'ΔSmix',
    'Ω':                            'Ω',
    'Ec':                           'Ec',
    'deltaP_E13 Electron affinity': 'ΔP_EA',
    'deltaP_热导率W/(mk)':           'ΔP_TC',
}

TSNE_PERPLEXITY    = 30
TSNE_LEARNING_RATE = 200
TSNE_N_ITER        = 1000
TSNE_RANDOM_STATE  = 42

PERPLEXITY_CANDIDATES = [5, 15, 30, 40, 50]

ROUND_COLORS = {
    0: '#AAAAAA',
    1: '#FF6B35',
    2: '#FFD700',
    3: '#00CC44',
    4: '#00BFFF',
    5: '#FF1493',
}
ROUND_LABELS = {
    1: 'Iteration #1',
    2: 'Iteration #2',
    3: 'Iteration #3',
    4: 'Iteration #4',
    5: 'Iteration #5',
}

KQ_CMAP  = 'RdYlGn'
KDE_CMAP = 'YlOrRd'

# ============================================================
# 0. 创建输出目录
# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("=" * 65)
print("t-SNE降维 + 主动学习迭代演化可视化")
print("=" * 65)
print(f"\n输出目录: {OUTPUT_DIR}")

# ============================================================
# 1. 读取数据
# ============================================================
print("\n读取数据...")
df = pd.read_excel(INPUT_FILE)
print(f"   总样本数 : {len(df)}")
print(f"   列名     : {list(df.columns)}")

df_origin = df[df['Round'] == 0].copy()
df_iter   = df[df['Round'] > 0].copy()

print(f"\n   原始样本 (Round=0): {len(df_origin)} 个")
for r in range(1, 6):
    n = len(df[df['Round'] == r])
    print(f"   Iteration #{r} (Round={r}): {n} 个")
print(f"\n   KQ 范围: [{df['KQ'].min():.2f}, {df['KQ'].max():.2f}]")

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURE_COLS].values)

# ============================================================
# 2. 图00：困惑度对比图（完全不变）
# ============================================================
print(f"\n生成图00 困惑度对比图（测试 {PERPLEXITY_CANDIDATES}）...")
print("   注意：t-SNE较慢，请耐心等待...")

fig0, axes0 = plt.subplots(
    1, len(PERPLEXITY_CANDIDATES),
    figsize=(5 * len(PERPLEXITY_CANDIDATES), 4.5)
)

perp_embeddings = {}

for idx, perp in enumerate(PERPLEXITY_CANDIDATES):
    print(f"   运行 perplexity={perp}...")
    tsne_tmp = TSNE(
        n_components=2,
        perplexity=perp,
        learning_rate=TSNE_LEARNING_RATE,
        n_iter=TSNE_N_ITER,
        random_state=TSNE_RANDOM_STATE,
        init='pca'
    )
    emb_tmp = tsne_tmp.fit_transform(X_scaled)
    perp_embeddings[perp] = emb_tmp

    ax = axes0[idx]
    ax.scatter(
        emb_tmp[df['Round'].values == 0, 0],
        emb_tmp[df['Round'].values == 0, 1],
        c=df.loc[df['Round'] == 0, 'KQ'].values,
        cmap=KQ_CMAP, s=20, alpha=0.6, edgecolors='none'
    )
    for r in range(1, 6):
        mask = df['Round'].values == r
        if mask.sum() > 0:
            ax.scatter(emb_tmp[mask, 0], emb_tmp[mask, 1],
                       c=ROUND_COLORS[r], s=120, marker='*',
                       edgecolors='black', linewidths=0.6, zorder=5)

    ax.set_title(f'perplexity = {perp}', fontsize=11, fontweight='bold')
    ax.set_xlabel('t-SNE 1', fontsize=9)
    if idx == 0:
        ax.set_ylabel('t-SNE 2', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.2, linestyle='--')

    if perp == TSNE_PERPLEXITY:
        for spine in ax.spines.values():
            spine.set_edgecolor('#D4522A')
            spine.set_linewidth(2.5)
        ax.set_title(f'perplexity = {perp}  ★ 主图使用',
                     fontsize=11, fontweight='bold', color='#D4522A')

fig0.suptitle(
    't-SNE Perplexity Comparison — Select the Best Parameter',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
out0 = os.path.join(OUTPUT_DIR, "tSNE_00_perplexity_comparison.png")
fig0.savefig(out0, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig0)
print(f"   已保存: {out0}")

# ============================================================
# 3. 主图 t-SNE（完全不变）
# ============================================================
print(f"\nt-SNE 主降维 (perplexity={TSNE_PERPLEXITY})...")
embedding = perp_embeddings[TSNE_PERPLEXITY]
df['tSNE_1'] = embedding[:, 0]
df['tSNE_2'] = embedding[:, 1]
df_origin = df[df['Round'] == 0].copy()
df_iter   = df[df['Round'] > 0].copy()
print(f"   完成  Dim1:[{embedding[:,0].min():.2f},{embedding[:,0].max():.2f}]"
      f"  Dim2:[{embedding[:,1].min():.2f},{embedding[:,1].max():.2f}]")

# ============================================================
# 4. 各轮重心（完全不变）
# ============================================================
centroids = {}
for r in range(1, 6):
    df_r = df[df['Round'] == r]
    if len(df_r) > 0:
        centroids[r] = {
            'x':       df_r['tSNE_1'].mean(),
            'y':       df_r['tSNE_2'].mean(),
            'kq_mean': df_r['KQ'].mean()
        }
centroid_rounds = sorted(centroids.keys())

print("\n   各轮重心:")
for r, c in centroids.items():
    print(f"      Round {r}: ({c['x']:.3f}, {c['y']:.3f})  平均KQ={c['kq_mean']:.2f}")

kq_min = df['KQ'].min()
kq_max = df['KQ'].max()
norm   = Normalize(vmin=kq_min, vmax=kq_max)

x_lim = (embedding[:,0].min()-3, embedding[:,0].max()+3)
y_lim = (embedding[:,1].min()-3, embedding[:,1].max()+3)
x_grid = np.linspace(x_lim[0], x_lim[1], 120)
y_grid = np.linspace(y_lim[0], y_lim[1], 120)
xx, yy  = np.meshgrid(x_grid, y_grid)
grid_pts = np.vstack([xx.flatten(), yy.flatten()])

high_kq_mask = df_origin['KQ'] >= np.percentile(df_origin['KQ'], 75)
hkq_x = df_origin.loc[high_kq_mask, 'tSNE_1'].mean()
hkq_y = df_origin.loc[high_kq_mask, 'tSNE_2'].mean()

# ============================================================
# 5. KDE 计算（完全不变）
# ============================================================
print("\n计算 KDE 密度...")
try:
    kde_origin = gaussian_kde(
        np.vstack([df_origin['tSNE_1'], df_origin['tSNE_2']]), bw_method=0.3
    )
    zz_origin = kde_origin(grid_pts).reshape(xx.shape)
except Exception as e:
    print(f"   原始KDE失败: {e}"); zz_origin = np.zeros_like(xx)

kde_grids = {}
for round_num in range(0, 6):
    df_cumul = df[df['Round'] <= round_num]
    try:
        kde_r = gaussian_kde(
            np.vstack([df_cumul['tSNE_1'], df_cumul['tSNE_2']]),
            bw_method=0.35
        )
        zz_r = kde_r(grid_pts).reshape(xx.shape)
    except Exception as e:
        print(f"   Round {round_num} KDE失败: {e}"); zz_r = np.zeros_like(xx)
    kde_grids[round_num] = zz_r
print("   KDE 计算完成")

# ============================================================
# 6. 导出 Excel（原有5个不变，新增2个）
# ============================================================
print("\n导出 Excel 数据（供 Origin 绘图）...")

# 6-1 t-SNE坐标（不变）
excel1_path = os.path.join(OUTPUT_DIR, "data_01_tSNE_coordinates.xlsx")
df[['Sample_ID','Round']+FEATURE_COLS+['KQ','tSNE_1','tSNE_2']].to_excel(
    excel1_path, index=False, sheet_name='tSNE坐标_图01_02_03主散点')
print(f"   {os.path.basename(excel1_path)}")

# 6-2 重心（不变）
excel2_path = os.path.join(OUTPUT_DIR, "data_02_centroids.xlsx")
pd.DataFrame([{
    'Round': r, 'Label': ROUND_LABELS[r],
    'Centroid_tSNE1': c['x'], 'Centroid_tSNE2': c['y'],
    'Mean_KQ': c['kq_mean'], 'Color': ROUND_COLORS[r]
} for r, c in centroids.items()]).to_excel(
    excel2_path, index=False, sheet_name='重心轨迹_图01_03箭头')
print(f"   {os.path.basename(excel2_path)}")

# 6-3 原始KDE（不变）
excel3_path = os.path.join(OUTPUT_DIR, "data_03_KDE_grid_origin.xlsx")
pd.DataFrame({
    'X_tSNE1': xx.flatten(), 'Y_tSNE2': yy.flatten(),
    'KDE_density': zz_origin.flatten()
}).to_excel(excel3_path, index=False, sheet_name='原始KDE_图01_03背景等高线')
print(f"   {os.path.basename(excel3_path)}")

# 6-4 各轮KDE（不变）
excel4_path = os.path.join(OUTPUT_DIR, "data_04_KDE_grid_round0to5.xlsx")
sheet_names_kde = [
    '图02_03_Training_data', '图02_03_After_Iter1', '图02_03_After_Iter2',
    '图02_03_After_Iter3',   '图02_03_After_Iter4', '图02_03_After_Iter5',
]
with pd.ExcelWriter(excel4_path, engine='openpyxl') as writer:
    for round_num in range(0, 6):
        df_cumul = df[df['Round'] <= round_num]
        pd.DataFrame({
            'X_tSNE1': xx.flatten(), 'Y_tSNE2': yy.flatten(),
            'KDE_density': kde_grids[round_num].flatten(),
            'n_samples': len(df_cumul)
        }).to_excel(writer, sheet_name=sheet_names_kde[round_num], index=False)
print(f"   {os.path.basename(excel4_path)}")

# 6-5 困惑度对比（不变）
excel5_path = os.path.join(OUTPUT_DIR, "data_05_perplexity_comparison.xlsx")
with pd.ExcelWriter(excel5_path, engine='openpyxl') as writer:
    for perp, emb in perp_embeddings.items():
        pd.DataFrame({
            'Sample_ID': df['Sample_ID'].values,
            'Round':     df['Round'].values,
            'KQ':        df['KQ'].values,
            'tSNE_1':    emb[:, 0], 'tSNE_2': emb[:, 1],
            'Is_main':   ['YES' if perp == TSNE_PERPLEXITY else 'no'] * len(df)
        }).to_excel(writer, sheet_name=f'图00_perplexity_{perp}', index=False)
print(f"   {os.path.basename(excel5_path)}")

# ── 6-6  KQ小提琴图数据（新增，对应图04）────────────────────
excel6_path = os.path.join(OUTPUT_DIR, "data_06_KQ_violin.xlsx")
kq_rows = []
for r in range(0, 6):
    df_r = df[df['Round'] == r]
    label = f'Round 0 (Training, n={len(df_r)})' if r == 0 \
            else f'Iter #{r} (n={len(df_r)})'
    for kq_val in df_r['KQ'].values:
        kq_rows.append({'Round': r, 'Round_label': label, 'KQ': kq_val})
kq_df = pd.DataFrame(kq_rows)

with pd.ExcelWriter(excel6_path, engine='openpyxl') as writer:
    # Sheet1：原始数据（每行一个样本）
    kq_df.to_excel(writer, sheet_name='图04_KQ原始数据', index=False)
    # Sheet2：统计汇总（均值/中位数/最大/最小/标准差）
    stats_rows = []
    for r in range(0, 6):
        sub = kq_df[kq_df['Round'] == r]['KQ']
        label = f'Round 0 (Training)' if r == 0 else f'Iter #{r}'
        stats_rows.append({
            'Round': r, 'Label': label,
            'Count': len(sub), 'Mean': sub.mean(),
            'Median': sub.median(), 'Std': sub.std(),
            'Min': sub.min(), 'Max': sub.max(),
            'Q25': sub.quantile(0.25), 'Q75': sub.quantile(0.75)
        })
    pd.DataFrame(stats_rows).to_excel(
        writer, sheet_name='图04_KQ统计汇总', index=False)
print(f"   {os.path.basename(excel6_path)}")

# ── 6-7  特征热力图数据（新增，对应图05）────────────────────
excel7_path = os.path.join(OUTPUT_DIR, "data_07_feature_heatmap.xlsx")

# 以Round0为基准计算Z-score偏移
baseline_mean = df[df['Round']==0][FEATURE_COLS].mean().values
baseline_std  = df[df['Round']==0][FEATURE_COLS].std().values + 1e-10
feat_short_list = [FEATURE_SHORT[f] for f in FEATURE_COLS]

heatmap_data = np.zeros((6, len(FEATURE_COLS)))
for r in range(6):
    df_r = df[df['Round'] == r]
    if len(df_r) > 0:
        row_mean = df_r[FEATURE_COLS].mean().values
        heatmap_data[r, :] = (row_mean - baseline_mean) / baseline_std

round_labels_heat = [
    'Round 0 (Training)' if r == 0 else f'Iter #{r}' for r in range(6)
]

with pd.ExcelWriter(excel7_path, engine='openpyxl') as writer:
    # Sheet1：Z-score热力图矩阵（6行×6列）
    df_heat = pd.DataFrame(
        heatmap_data,
        index=round_labels_heat,
        columns=feat_short_list
    )
    df_heat.index.name = 'Round'
    df_heat.to_excel(writer, sheet_name='图05_热力图Z-score矩阵')

    # Sheet2：各轮各特征原始均值
    mean_rows = []
    for r in range(6):
        df_r = df[df['Round'] == r]
        row = {'Round': round_labels_heat[r]}
        for feat in FEATURE_COLS:
            row[FEATURE_SHORT[feat]+'_mean'] = df_r[feat].mean()
        mean_rows.append(row)
    pd.DataFrame(mean_rows).to_excel(
        writer, sheet_name='图05_各轮特征均值', index=False)

    # Sheet3：各轮各特征原始标准差
    std_rows = []
    for r in range(6):
        df_r = df[df['Round'] == r]
        row = {'Round': round_labels_heat[r]}
        for feat in FEATURE_COLS:
            row[FEATURE_SHORT[feat]+'_std'] = df_r[feat].std()
        std_rows.append(row)
    pd.DataFrame(std_rows).to_excel(
        writer, sheet_name='图05_各轮特征标准差', index=False)

    # Sheet4：KQ均值和最大值（右侧折线图数据）
    kq_evo_rows = []
    for r in range(6):
        df_r = df[df['Round'] == r]
        kq_evo_rows.append({
            'Round': r,
            'Label': round_labels_heat[r],
            'Mean_KQ': df_r['KQ'].mean(),
            'Max_KQ':  df_r['KQ'].max(),
            'Min_KQ':  df_r['KQ'].min(),
            'Std_KQ':  df_r['KQ'].std(),
        })
    pd.DataFrame(kq_evo_rows).to_excel(
        writer, sheet_name='图05_KQ演化折线数据', index=False)
print(f"   {os.path.basename(excel7_path)}")

# ============================================================
# 7. 绘图辅助函数（完全不变）
# ============================================================
DIM1_LABEL = 't-SNE Dimension 1'
DIM2_LABEL = 't-SNE Dimension 2'

panel_titles = [
    'Training\ndata', 'After\nIteration #1', 'After\nIteration #2',
    'After\nIteration #3', 'After\nIteration #4', 'After\nIteration #5'
]


def draw_main_scatter(ax, show_title=True):
    sc = ax.scatter(
        df_origin['tSNE_1'], df_origin['tSNE_2'],
        c=df_origin['KQ'], cmap=KQ_CMAP,
        vmin=kq_min, vmax=kq_max,
        s=60, alpha=0.5, edgecolors='gray', linewidths=0.3,
        zorder=2, label=f'Training data (n={len(df_origin)})'
    )
    try:
        ax.contour(xx, yy, zz_origin, levels=5,
                   colors='gray', alpha=0.22, linewidths=0.8, zorder=1)
    except Exception:
        pass
    for r in range(1, 6):
        df_r = df[df['Round'] == r]
        if len(df_r) == 0: continue
        ax.scatter(df_r['tSNE_1'], df_r['tSNE_2'],
                   c=ROUND_COLORS[r], s=260, marker='*',
                   edgecolors='black', linewidths=0.8,
                   zorder=5, label=ROUND_LABELS[r])
        for _, row in df_r.iterrows():
            ax.annotate(f'{row["KQ"]:.1f}',
                xy=(row['tSNE_1'], row['tSNE_2']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=7.5, color=ROUND_COLORS[r],
                fontweight='bold', zorder=6)
    for i in range(len(centroid_rounds)-1):
        rc = centroid_rounds[i];  rn = centroid_rounds[i+1]
        cc = centroids[rc];       cn = centroids[rn]
        ax.annotate('', xy=(cn['x'],cn['y']), xytext=(cc['x'],cc['y']),
            arrowprops=dict(arrowstyle='->,head_width=0.32,head_length=0.22',
                color='black', lw=2.1, connectionstyle='arc3,rad=0.15'),
            zorder=7)
    for r, c in centroids.items():
        ax.plot(c['x'], c['y'], marker='D', markersize=10,
                color=ROUND_COLORS[r], markeredgecolor='black',
                markeredgewidth=1.0, zorder=8)
    if high_kq_mask.sum() > 3:
        ax.annotate('High $K_Q$ region', xy=(hkq_x, hkq_y),
            fontsize=11, color='darkgreen', fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='lightgreen', alpha=0.4), zorder=9)
    ax.text(0.98, 0.02, f'perplexity = {TSNE_PERPLEXITY}',
            transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
            color='gray',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      alpha=0.7, edgecolor='lightgray'))
    ax.set_xlabel(DIM1_LABEL, fontsize=12, fontweight='bold')
    ax.set_ylabel(DIM2_LABEL, fontsize=12, fontweight='bold')
    ax.set_xlim(x_lim); ax.set_ylim(y_lim)
    if show_title:
        ax.set_title(
            'Active Learning Exploration Trajectory in t-SNE Feature Space',
            fontsize=13, fontweight='bold', pad=10)
    ax.legend(loc='upper left', fontsize=9.5, framealpha=0.85, edgecolor='gray')
    ax.grid(True, alpha=0.2, linestyle='--')
    ax.tick_params(labelsize=10)
    return sc


def draw_kde_panel(ax, round_num, col_idx):
    df_cumul = df[df['Round'] <= round_num]
    zz_r = kde_grids.get(round_num, np.zeros_like(xx))
    ax.contourf(xx, yy, zz_r, levels=15, cmap=KDE_CMAP, alpha=0.85)
    ax.contour(xx, yy, zz_r,  levels=8,  colors='white', alpha=0.4, linewidths=0.5)
    ax.scatter(df_origin['tSNE_1'], df_origin['tSNE_2'],
               c='white', s=7, alpha=0.35, edgecolors='none', zorder=3)
    for r in range(1, round_num+1):
        df_r = df[df['Round'] == r]
        if len(df_r) > 0:
            ax.scatter(df_r['tSNE_1'], df_r['tSNE_2'],
                       c=ROUND_COLORS[r], s=110, marker='*',
                       edgecolors='black', linewidths=0.5, zorder=5)
    ax.text(0.05, 0.05, f'n={len(df_cumul)}', transform=ax.transAxes,
            fontsize=8.5, color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))
    ax.set_xlim(x_lim); ax.set_ylim(y_lim)
    ax.set_title(panel_titles[col_idx], fontsize=9.5, fontweight='bold', pad=5)
    ax.set_xlabel('t-SNE 1', fontsize=8.5)
    if col_idx == 0:
        ax.set_ylabel('t-SNE 2', fontsize=8.5)
    else:
        ax.set_yticklabels([])
    ax.tick_params(labelsize=7.5)

# ============================================================
# 8. 图01：主散点图（完全不变）
# ============================================================
print("\n生成图01 主散点图...")
fig1, ax1 = plt.subplots(figsize=(10, 8))
sc1 = draw_main_scatter(ax1, show_title=True)
cbar1 = plt.colorbar(sc1, ax=ax1, pad=0.02)
cbar1.set_label('$K_Q$ (MPa·m$^{1/2}$)', fontsize=12, fontweight='bold')
cbar1.ax.tick_params(labelsize=10)
plt.tight_layout()
out1 = os.path.join(OUTPUT_DIR, "tSNE_01_main_scatter.png")
fig1.savefig(out1, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig1)
print(f"   已保存: {out1}")

# ============================================================
# 9. 图02：密度演化图（完全不变）
# ============================================================
print("\n生成图02 密度演化图（1×6）...")
fig2, axes2 = plt.subplots(1, 6, figsize=(22, 4.2))
for col_idx, rn in enumerate(range(0, 6)):
    draw_kde_panel(axes2[col_idx], rn, col_idx)
fig2.suptitle(
    'Exploration Space Evolution during Active Learning Iterations (t-SNE Space)',
    fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
out2 = os.path.join(OUTPUT_DIR, "tSNE_02_density_evolution.png")
fig2.savefig(out2, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig2)
print(f"   已保存: {out2}")

# ============================================================
# 10. 图03：合并版（完全不变）
# ============================================================
print("\n生成图03 合并版最终图（论文用）...")
fig3 = plt.figure(figsize=(18, 15))
ax_top = fig3.add_axes([0.05, 0.42, 0.86, 0.54])
sc_top = draw_main_scatter(ax_top, show_title=False)
ax_top.set_title(
    'Active Learning Exploration Trajectory in t-SNE Feature Space',
    fontsize=13, fontweight='bold', pad=10, loc='left')
cbar_ax = fig3.add_axes([0.928, 0.42, 0.013, 0.54])
sm = ScalarMappable(cmap=KQ_CMAP, norm=norm)
sm.set_array([])
cb = fig3.colorbar(sm, cax=cbar_ax)
cb.set_label('$K_Q$ (MPa·m$^{1/2}$)', fontsize=12, fontweight='bold')
cb.ax.tick_params(labelsize=10)
n_panels = 6; panel_w = 0.86/n_panels; panel_gap = 0.005
for col_idx, rn in enumerate(range(0, 6)):
    left = 0.05 + col_idx*(panel_w+panel_gap)
    ax_bot = fig3.add_axes([left, 0.03, panel_w-panel_gap, 0.34])
    draw_kde_panel(ax_bot, rn, col_idx)
out3 = os.path.join(OUTPUT_DIR, "tSNE_03_FINAL.png")
fig3.savefig(out3, dpi=300, bbox_inches='tight',
             facecolor='white', edgecolor='none')
plt.close(fig3)
print(f"   已保存: {out3}")

# ============================================================
# 11. 图04：KQ 小提琴图（新增）
# ============================================================
print("\n生成图04 KQ小提琴图...")
fig4, ax4 = plt.subplots(figsize=(11, 6))

violin_data   = []
violin_colors = []
xtick_labels  = []

for r in range(0, 6):
    df_r = df[df['Round'] == r]
    if len(df_r) == 0:
        continue
    violin_data.append(df_r['KQ'].values)
    violin_colors.append(ROUND_COLORS[r])
    xtick_labels.append(
        f'Round 0\n(Training)\nn={len(df_r)}' if r == 0
        else f'Iter #{r}\n(n={len(df_r)})'
    )

positions = range(len(violin_data))
parts = ax4.violinplot(
    violin_data, positions=positions,
    showmeans=True, showmedians=True,
    showextrema=True, widths=0.7
)

for pc, col in zip(parts['bodies'], violin_colors):
    pc.set_facecolor(col)
    pc.set_alpha(0.55)
    pc.set_edgecolor('black')
    pc.set_linewidth(0.8)

parts['cmeans'].set_color('black')
parts['cmeans'].set_linewidth(2.2)
parts['cmedians'].set_color('navy')
parts['cmedians'].set_linewidth(1.8)
for key in ['cbars', 'cmins', 'cmaxes']:
    parts[key].set_color('black')
    parts[key].set_linewidth(1.0)

# 叠加散点（jitter）
np.random.seed(42)
for i, (data, col) in enumerate(zip(violin_data, violin_colors)):
    jitter = np.random.uniform(-0.1, 0.1, len(data))
    ax4.scatter(np.full(len(data), i) + jitter, data,
                color=col, s=30, alpha=0.75,
                edgecolors='black', linewidths=0.4, zorder=5)

# 标注均值和最大值
for i, data in enumerate(violin_data):
    ax4.text(i, np.max(data)+0.18,
             f'max={np.max(data):.1f}',
             ha='center', va='bottom', fontsize=8.5,
             color='#993300', fontweight='bold')
    ax4.text(i, np.mean(data)-0.55,
             f'μ={np.mean(data):.1f}',
             ha='center', va='top', fontsize=8.5,
             color='black', fontweight='bold')

ax4.set_xticks(positions)
ax4.set_xticklabels(xtick_labels, fontsize=10)
ax4.set_ylabel('$K_Q$ (MPa·m$^{1/2}$)', fontsize=12, fontweight='bold')
ax4.set_title(
    '$K_Q$ Distribution across Active Learning Iterations',
    fontsize=13, fontweight='bold', pad=10)
ax4.grid(True, axis='y', alpha=0.25, linestyle='--')
ax4.tick_params(labelsize=10)

legend_elements = [
    Line2D([0],[0], color='black', linewidth=2.2, label='Mean'),
    Line2D([0],[0], color='navy',  linewidth=1.8, label='Median'),
]
ax4.legend(handles=legend_elements, fontsize=10,
           loc='upper left', framealpha=0.85)

plt.tight_layout()
out4 = os.path.join(OUTPUT_DIR, "tSNE_04_KQ_violin.png")
fig4.savefig(out4, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig4)
print(f"   已保存: {out4}")

# ============================================================
# 12. 图05：特征分布热力图（新增）
# ============================================================
print("\n生成图05 特征分布热力图...")
fig5, axes5 = plt.subplots(1, 2, figsize=(14, 5.5),
                            gridspec_kw={'width_ratios': [2, 1]})

round_labels_heat = [
    'Round 0\n(Training)' if r == 0 else f'Iter #{r}'
    for r in range(6)
]
vmax = np.abs(heatmap_data).max()

# ── 左图：Z-score热力图 ──
cmap_heat = plt.cm.RdBu_r
im = axes5[0].imshow(heatmap_data, cmap=cmap_heat,
                      vmin=-vmax, vmax=vmax, aspect='auto')
axes5[0].set_xticks(range(len(FEATURE_COLS)))
axes5[0].set_xticklabels(feat_short_list, fontsize=12, fontweight='bold')
axes5[0].set_yticks(range(6))
axes5[0].set_yticklabels(round_labels_heat, fontsize=10)
axes5[0].set_title(
    'Feature Distribution Shift\n(Z-score relative to training data)',
    fontsize=12, fontweight='bold', pad=8)

for i in range(6):
    for j in range(len(FEATURE_COLS)):
        val = heatmap_data[i, j]
        text_color = 'white' if abs(val) > vmax*0.55 else 'black'
        axes5[0].text(j, i, f'{val:.2f}',
                      ha='center', va='center',
                      fontsize=10, color=text_color, fontweight='bold')

cbar_h = plt.colorbar(im, ax=axes5[0], pad=0.02, shrink=0.9)
cbar_h.set_label('Z-score', fontsize=10, fontweight='bold')
cbar_h.ax.tick_params(labelsize=9)

# ── 右图：KQ均值+最大值折线图 ──
round_mean_kq = [df[df['Round']==r]['KQ'].mean() for r in range(6)]
round_max_kq  = [df[df['Round']==r]['KQ'].max()  for r in range(6)]
x_pos = range(6)

axes5[1].plot(x_pos, round_mean_kq, 'o-',
              color='#3A5FA8', linewidth=2.2, markersize=8,
              markeredgecolor='white', markeredgewidth=1.2,
              label='Mean $K_Q$', zorder=4)
axes5[1].plot(x_pos, round_max_kq, 's--',
              color='#D4522A', linewidth=1.8, markersize=7,
              markeredgecolor='white', markeredgewidth=1.2,
              label='Max $K_Q$', zorder=4)

for xi, mv in zip(x_pos, round_mean_kq):
    axes5[1].text(xi, mv-0.35, f'{mv:.1f}',
                  ha='center', va='top', fontsize=8.5, color='#3A5FA8',
                  fontweight='bold')
for xi, xv in zip(x_pos, round_max_kq):
    axes5[1].text(xi, xv+0.2, f'{xv:.1f}',
                  ha='center', va='bottom', fontsize=8.5, color='#D4522A',
                  fontweight='bold')

axes5[1].set_xticks(x_pos)
axes5[1].set_xticklabels(
    ['R0\n(Train)','Iter#1','Iter#2','Iter#3','Iter#4','Iter#5'],
    fontsize=9)
axes5[1].set_ylabel('$K_Q$ (MPa·m$^{1/2}$)', fontsize=11, fontweight='bold')
axes5[1].set_title('$K_Q$ Evolution\nper Iteration',
                   fontsize=12, fontweight='bold', pad=8)
axes5[1].legend(fontsize=9.5, framealpha=0.85, loc='upper left')
axes5[1].grid(True, alpha=0.25, linestyle='--')
axes5[1].tick_params(labelsize=9)

plt.tight_layout()
out5 = os.path.join(OUTPUT_DIR, "tSNE_05_feature_heatmap.png")
fig5.savefig(out5, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig5)
print(f"   已保存: {out5}")

# ============================================================
# 13. 汇总报告
# ============================================================
print("\n" + "=" * 65)
print("全部完成！")
print("=" * 65)

print(f"\n图片文件（共6张）:")
print(f"   {os.path.basename(out0)}  <- 困惑度对比图")
print(f"   {os.path.basename(out1)}  <- 主散点图")
print(f"   {os.path.basename(out2)}  <- 密度演化图")
print(f"   {os.path.basename(out3)}  <- 合并版（论文直接使用）")
print(f"   {os.path.basename(out4)}  <- KQ小提琴图（新增）")
print(f"   {os.path.basename(out5)}  <- 特征分布热力图（新增）")

print(f"\nExcel数据文件（共7个）:")
print(f"   data_01 — 对应图01/02/03 t-SNE坐标")
print(f"   data_02 — 对应图01/03 重心轨迹")
print(f"   data_03 — 对应图01/03 背景KDE")
print(f"   data_04 — 对应图02/03 密度演化")
print(f"   data_05 — 对应图00 困惑度对比")
print(f"   data_06 — 对应图04 KQ小提琴（含原始数据+统计汇总）")
print(f"   data_07 — 对应图05 特征热力图（含Z-score矩阵+均值+标准差+KQ演化）")

print(f"\nt-SNE 参数（论文写作用）:")
print(f"   perplexity    = {TSNE_PERPLEXITY}")
print(f"   learning_rate = {TSNE_LEARNING_RATE}")
print(f"   n_iter        = {TSNE_N_ITER}")
print(f"   init          = pca")
print(f"   random_state  = {TSNE_RANDOM_STATE}")

print(f"\n数据统计:")
print(f"   总样本数  : {len(df)}")
print(f"   原始样本  : {len(df_origin)}")
print(f"   迭代新增  : {len(df_iter)}")
print(f"   原始最高KQ: {df_origin['KQ'].max():.2f}")
print(f"   迭代最高KQ: {df_iter['KQ'].max():.2f}")
print(f"   KQ 提升   : +{df_iter['KQ'].max()-df_origin['KQ'].max():.2f}")
print("=" * 65)

t-SNE降维 + 主动学习迭代演化可视化

输出目录: D:\博一下\第一章主动学习\主动学习结果_v7.6_方案D_LogEI_SHAP全程引导版\迭代可视化_tSNE2

读取数据...
   总样本数 : 217
   列名     : ['Sample_ID', 'Round', 'Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf', 'Zr', 'Mo', 'V', 'W', 'Ta', 'Sn', 'x', 'ΔSmix', 'Ω', 'Ec', 'deltaP_E13 Electron affinity', 'deltaP_热导率W/(mk)', 'KQ']

   原始样本 (Round=0): 202 个
   Iteration #1 (Round=1): 3 个
   Iteration #2 (Round=2): 3 个
   Iteration #3 (Round=3): 3 个
   Iteration #4 (Round=4): 3 个
   Iteration #5 (Round=5): 3 个

   KQ 范围: [3.40, 21.09]

生成图00 困惑度对比图（测试 [5, 15, 30, 40, 50]）...
   注意：t-SNE较慢，请耐心等待...
   运行 perplexity=5...
   运行 perplexity=15...
   运行 perplexity=30...
   运行 perplexity=40...
   运行 perplexity=50...
   已保存: D:\博一下\第一章主动学习\主动学习结果_v7.6_方案D_LogEI_SHAP全程引导版\迭代可视化_tSNE2\tSNE_00_perplexity_comparison.png

t-SNE 主降维 (perplexity=30)...
   完成  Dim1:[-18.89,11.61]  Dim2:[-17.85,17.05]

   各轮重心:
      Round 1: (7.687, 14.561)  平均KQ=18.41
      Round 2: (9.045, 14.993)  平均KQ=19.63
      Round 3: (6.632, 13.491)  平均KQ=19.93